In [ ]:
# Step 1: Defining a Tool with Structured Metadata
# A basic tool definition in Python might look like this:

def list_python_files():
    """Returns a list of all Python files in the src/ directory."""
    return [f for f in os.listdir("src") if f.endswith(".py")]

In [ ]:
# For example, a tool that reads a file should specify that it expects a file_path parameter of type string. JSON Schema allows us to express this in a standardized way:

{
  "tool_name": "read_file",
  "description": "Reads the content of a specified file.",
  "parameters": {
    "type": "object",
    "properties": {
      "file_path": { "type": "string" }
    },
    "required": ["file_path"]
  }
}

In [ ]:
# Similarly, a tool for writing documentation should define that it requires a file_name and content

{
  "tool_name": "write_doc_file",
  "description": "Writes a documentation file to the docs/ directory.",
  "parameters": {
    "type": "object",
    "properties": {
      "file_name": { "type": "string" },
      "content": { "type": "string" }
    },
    "required": ["file_name", "content"]
  }
}

In [2]:
""" By providing a JSON Schema for each tool:

The AI can Recognize the tool’s purpose.
The AI / Environment interface can validate input parameters before execution.
It may look strange that multiple parameters to a function are represented as an object. When we are getting the agent to output a tool / action selection, we are going to want it to output something like this:
"""

{
  "tool_name": "read_file",
  "args": {
    "file_path": "src/file.py"
  }
}

{'tool_name': 'read_file', 'args': {'file_path': 'src/file.py'}}

In [3]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
#
# You will also want to add some sample files to the "Files" (folder icon)
# on the left. When the agent asks you what to do, start with something
# simplie like "tell me what files are in this directory"
#
import os
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key

In [9]:
import json
import os
import sys
from litellm import completion
from typing import List, Dict

def extract_markdown_block(response: str, block_type: str = "json") -> str:
    """Extract code block from response"""

    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()

    if code_block.startswith(block_type):
        code_block = code_block[len(block_type):].strip()

    return code_block


def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get a response."""
    response = completion(
        model="openai/gpt-4o",
        messages=messages,
        max_tokens=1024
    )
    # print(f"Unfiler response: {response}")
    # if response.choices[0].message.tool_calls[0]:
    #   tool = response.choices[0].message.tool_calls[0]
    #   print(f"Unfiltered tool: {tool}")
    #   tool_name = tool.function.name
    #   print(f"Unfiltered tool_name: {tool_name}")
    #   tool_args = json.loads(tool.function.arguments)
    #   print(f"Unfiltered tool_args: {tool_args}")

    return response.choices[0].message.content.strip()

def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

def list_files() -> List[str]:
    """List files in the current directory."""
    return os.listdir(".")

def read_file(file_name: str) -> str:
    """Read a file's contents."""
    try:
        with open(file_name, "r") as file:
            return file.read()
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

# Define system instructions (Agent Rules)
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:

```json
{
    "list_files": {
        "description": "Lists all files in the current directory.",
        "parameters": {}
    },
    "read_file": {
        "description": "Reads the content of a file.",
        "parameters": {
            "file_name": {
                "type": "string",
                "description": "The name of the file to read."
            }
        }
    },
    "terminate": {
        "description": "Ends the agent loop and provides a summary of the task.",
        "parameters": {
            "message": {
                "type": "string",
                "description": "Summary message to return to the user."
            }
        }
    }
}
```

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.

Important!!! Every response MUST have an action.
You must ALWAYS respond in this format:

<Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```"""
}]

# Initialize agent parameters
iterations = 0
max_iterations = 10

user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]

# The Agent Loop
while iterations < max_iterations:
    # 1. Construct prompt: Combine agent rules with memory
    prompt = agent_rules + memory

    # 2. Generate response from LLM
    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    # 3. Parse response to determine action
    action = parse_action(response)
    print(f"Agent action response: {action}")

    # 4. Execute action and get result
    result = "Action executed"

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"Action result: {result}")

    # 5. Update memory with response and results
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    # 6. Check termination condition
    if action["tool_name"] == "terminate":
        break

    iterations += 1

# tool = response.choices[0].message.tool_calls[0]
# tool_name = tool.function.name
# tool_args = json.loads(tool.function.arguments)

What would you like me to do? symmary of the python file
Agent thinking...
Agent response: To provide a summary of the Python file, I first need to list the files in the current directory to identify the Python files available. Let's start with that.

```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Agent action response: {'tool_name': 'list_files', 'args': {}}
Action result: {'result': ['.config', 'convert_heic.py', 'sample_data']}
Agent thinking...
Agent response: From the list of files available, I see there is a Python file named `convert_heic.py`. I will now read the contents of this file to provide a summary.

```action
{
    "tool_name": "read_file",
    "args": {
        "file_name": "convert_heic.py"
    }
}
```
Agent action response: {'tool_name': 'read_file', 'args': {'file_name': 'convert_heic.py'}}
Action result: {'result': '#!/usr/bin/env python3\n"""\nHEIC to JPG Converter\nConverts single or bulk HEIC images to JPG format.\n"""\n\nimport os\nimport sys\ni